# Export the pre-HFNO model for external validation in the Dutch Data Warehouse (DDW)

The DDW lives on a remote server; MIMIC-IV patient data must not be copied there. This notebook therefore exports
only **trained model files and summary statistics** (no patient rows) to `ddw_export/`, which is copied to the DDW
folder (`ddw_validation_original/mimic_model/`).

Two models, both XGBoost with the settings of Model 5 (pre-HFNO only) in notebook 3, trained on the full MIMIC cohort:

* **full** - all Model 5 predictors. Predictors that do not exist in the DDW are filled with the MIMIC median in DDW
  (as in the original manuscript).
* **transport** - retrained on MIMIC using only predictors that are available in the DDW. This is the cleaner external
  validation, because no predictor is replaced by a constant.

For reference, the internal (MIMIC held-out test set) AUROC of the transport model is reported, so that the loss from
dropping predictors can be separated from the loss due to the population shift.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn import metrics
from sklearn.model_selection import train_test_split

OUT = 'ddw_export'
os.makedirs(OUT, exist_ok=True)
TEST_SPLIT_FRAC, SEED = 0.15, 42
XGB_PARAMS = dict(objective='binary:logistic', random_state=SEED, max_depth=4, reg_alpha=2, reg_lambda=4,
                  min_child_weight=15, n_estimators=100, learning_rate=0.01, subsample=0.5)   # = xgb_s1, notebook 3

In [ ]:
df = pd.read_csv('./filtered_data/mimiciv_tabledata_imputed.csv')
raw = pd.read_csv('./filtered_data/mimiciv_tabledata.csv')           # non-imputed, for medians

data = df.iloc[:, 5:].drop(columns=['intubated', 'died', 'inhosp_mortality'])
data = data.drop(columns=['flow_rate_2h', 'fio2_2h', 'rox_2h'])      # recorded after HFNO start (as in notebook 3)
y = data.pop('hfno_failure').to_numpy()
FEATURES_FULL = data.columns.tolist()
print(len(FEATURES_FULL), 'predictors (Model 5):', FEATURES_FULL)

### Predictors available in the DDW

Based on the DDW extraction notebooks. Missing in DDW: myocardial infarction, congestive heart failure, peripheral
vascular disease, cerebrovascular disease, liver disease, renal disease, malignancy, ethnicity, PTT, MCH, MCHC, RDW.
Vasopressor use and fluid balance depend on DDW tables that still have to be confirmed on the server; they are left out
of the transport model by default (set `INCLUDE_UNCERTAIN = True` and rerun if the DDW notebooks confirm them).

In [ ]:
INCLUDE_UNCERTAIN = False
DDW_AVAILABLE = [
    'gender', 'admission_age', 'weight_admit', 'sepsis3', 'chronic_pulmonary_disease', 'diabetes', 'apsiii',
    'gcs_binned_Severe (3-8)', 'gcs_binned_Moderate (9-12)', 'gcs_binned_Mild (13-14)', 'gcs_binned_Normal (15)',
    'heart_rate_mean_last24h', 'mbp_mean_last24h', 'resp_rate_mean_last24h', 'temperature_mean_last24h',
    'spo2_mean_last24h', 'spo2_last', 'glucose_mean_last24h',
    'po2_last', 'pco2_last', 'ph_last', 'creatinine_last', 'platelet_last', 'potassium_last', 'wbc_last',
    'calcium_last', 'sodium_last', 'hemoglobin_last', 'aniongap_last', 'inr_last',
]
UNCERTAIN = ['vp_last6h', 'fluidbalance_24hr']
if INCLUDE_UNCERTAIN:
    DDW_AVAILABLE += UNCERTAIN
FEATURES_TRANSPORT = [f for f in FEATURES_FULL if f in DDW_AVAILABLE]
missing_in_ddw = [f for f in FEATURES_FULL if f not in DDW_AVAILABLE]
assert set(DDW_AVAILABLE) <= set(FEATURES_FULL), set(DDW_AVAILABLE) - set(FEATURES_FULL)
print(f'transport model: {len(FEATURES_TRANSPORT)} predictors; not available in DDW: {missing_in_ddw}')

### Internal performance on the MIMIC held-out test set (same split as notebook 3)

In [ ]:
idx_tr, idx_te = train_test_split(np.arange(len(y)), test_size=TEST_SPLIT_FRAC, random_state=SEED)

def fit(features, idx):
    X = data[features].to_numpy()
    ytr = y[idx]
    m = xgb.XGBClassifier(**XGB_PARAMS, scale_pos_weight=(1 - ytr.mean()) / ytr.mean())
    return m.fit(X[idx], ytr)

internal = {}
for name, feats in [('full', FEATURES_FULL), ('transport', FEATURES_TRANSPORT)]:
    m = fit(feats, idx_tr)
    internal[name] = metrics.roc_auc_score(y[idx_te], m.predict_proba(data[feats].to_numpy()[idx_te])[:, 1])
    print(f'{name:9s}: internal test AUROC {internal[name]:.3f}')

### Final models (trained on the full MIMIC cohort) and export

In [ ]:
UNITS = {   # units of the MIMIC predictors; the DDW notebooks convert to these
    'admission_age': 'years', 'weight_admit': 'kg', 'apsiii': 'points',
    'heart_rate_mean_last24h': '/min', 'mbp_mean_last24h': 'mmHg', 'resp_rate_mean_last24h': '/min',
    'temperature_mean_last24h': 'degC', 'spo2_mean_last24h': '%', 'spo2_last': '%', 'glucose_mean_last24h': 'mg/dL',
    'po2_last': 'mmHg', 'pco2_last': 'mmHg', 'ph_last': '', 'creatinine_last': 'mg/dL', 'platelet_last': '10^9/L',
    'potassium_last': 'mmol/L', 'wbc_last': '10^9/L', 'calcium_last': 'mg/dL', 'sodium_last': 'mmol/L',
    'hemoglobin_last': 'g/dL', 'aniongap_last': 'mmol/L', 'inr_last': '', 'fluidbalance_24hr': 'mL',
    'ptt_last': 's', 'mch_last': 'pg', 'mchc_last': 'g/dL', 'rdw_last': '%',
}
all_idx = np.arange(len(y))
medians = {}
for f in FEATURES_FULL:
    col = raw[f] if f in raw.columns and pd.api.types.is_numeric_dtype(raw[f]) else data[f]
    medians[f] = float(np.nanmedian(col))

for name, feats in [('full', FEATURES_FULL), ('transport', FEATURES_TRANSPORT)]:
    m = fit(feats, all_idx)
    m.save_model(f'{OUT}/model_{name}.json')
    spec = {'model': f'model_{name}.json', 'features': feats, 'feature_order_matters': True,
            'units': {f: UNITS.get(f, 'binary (0/1)') for f in feats},
            'mimic_medians': {f: medians[f] for f in feats},
            'mimic_internal_test_auroc': round(internal[name], 3),
            'n_mimic': int(len(y)), 'n_failures_mimic': int(y.sum()),
            'xgb_params': XGB_PARAMS,
            'notes': 'GCS bins follow MIMIC: Severe 3-8, Moderate 9-12, Mild 13-14, Normal 15. gender = 1 for male.'}
    json.dump(spec, open(f'{OUT}/model_{name}_spec.json', 'w'), indent=1)
    print(f'saved {OUT}/model_{name}.json and spec ({len(feats)} predictors)')